# 4. Harmonização Histórica via Processamento de Linguagem Natural (NLP)

Este notebook implementa a inteligência necessária para resolver o **Drift Semântico** das colunas do INEP ao longo das décadas.
Em vez de depender de códigos hardcoded (ex: `Q024` ou `Q021`), utilizamos um modelo de **Sentence Transformers (SBERT)** para ler a descrição das perguntas (ex: *Na sua residência tem computador?* vs *Em sua casa, existe computador?*) e pareá-las usando similaridade matemática (Cosine Similarity).

In [1]:
import pandas as pd
import numpy as np
import glob
import os
import re
from sentence_transformers import SentenceTransformer, util
import torch

# 1. Carregar o Modelo NLP
print("Carregando o modelo SBERT (pode demorar alguns segundos na primeira vez)...")
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
print("Modelo carregado com sucesso!")


/home/saulo/mba/tcc/enem/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Carregando o modelo SBERT (pode demorar alguns segundos na primeira vez)...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 843.32it/s]


Modelo carregado com sucesso!


In [2]:
# 2. Fazer o parse do dicionario_features.md (Padrão Ouro de 2023)
path_dic_md = 'notebooks/dicionario_features.md'
if not os.path.exists(path_dic_md):
    path_dic_md = 'dicionario_features.md'

with open(path_dic_md, 'r', encoding='utf-8') as f:
    content = f.read()

features_base = {}
blocks = re.split(r'={40}', content)
for i in range(len(blocks)):
    if 'FEATURE:' in blocks[i]:
        feat_name = blocks[i].replace('FEATURE:', '').strip()
        if i+1 < len(blocks):
            details = blocks[i+1].strip()
            desc_match = re.search(r'Descrição:\s*(.*?)(?=\nTamanho:)', details, re.DOTALL)
            if desc_match:
                features_base[feat_name] = desc_match.group(1).replace('\n', ' ').strip()

print(f"Features Base Extraídas ({len(features_base)}):")
for k, v in features_base.items():
    print(f" - {k}: {v}")

# Gerar Embeddings das features base
base_feats = list(features_base.keys())
base_descs = [features_base[f] for f in base_feats]
base_embs = model.encode(base_descs, convert_to_tensor=True)


Features Base Extraídas (6):
 - Q006: Qual é a renda mensal de sua família? (Some a sua renda com a dos seus familiares.)
 - Q024: Na sua residência tem computador?
 - TP_ANO_CONCLUIU: Ano de Conclusão do Ensino Médio
 - TP_LINGUA: Língua Estrangeira
 - TP_SEXO: Sexo
 - Q002: Até que série sua mãe, ou a mulher responsável por você, estudou?


In [3]:
# Função robusta para extrair TODAS as descrições de um arquivo de dicionário XLSX
def parse_dicionario_total(path):
    try:
        xls = pd.ExcelFile(path, engine='openpyxl')
        sheet = next((s for s in xls.sheet_names if 'PARTICIPANTES' in s.upper() or 'MICRODADOS' in s.upper()), xls.sheet_names[0])
        df = pd.read_excel(path, sheet_name=sheet, header=None, engine='openpyxl')
        header_row = next((i for i, row in df.iterrows() if 'NOME DA VARIÁVEL' in str(row.values) or 'NOME' in str(row.values).upper()), None)
        if header_row is None:
            return {}
        df2 = pd.read_excel(path, sheet_name=sheet, header=header_row, engine='openpyxl')
        df2.columns = [str(c).strip() for c in df2.columns]
        
        col_nome = [c for c in df2.columns if 'NOME' in c.upper()]
        col_desc = [c for c in df2.columns if 'DESCRI' in c.upper()]
        
        if not col_nome or not col_desc:
            return {}
            
        df2 = df2[[col_nome[0], col_desc[0]]].dropna(subset=[col_nome[0]])
        return dict(zip(df2[col_nome[0]].astype(str).str.strip(), df2[col_desc[0]].astype(str).str.strip()))
    except Exception as e:
        print(f"Erro em {path}: {e}")
        return {}


In [4]:
# 3. Iterar sobre todos os anos (2009 a 2025)
RAW_DATA_DIR = '../data/raw'
anos_paths = sorted(glob.glob(os.path.join(RAW_DATA_DIR, '*/microdados_enem_*/DICIONÁRIO/Dicion*.xlsx')))

historico_drift = []

for path in anos_paths:
    match = re.search(r'enem_(\d{4})', path)
    ano = match.group(1) if match else "Desconhecido"
    
    if ano == "2023":
        continue # O ano de 2023 é o nosso padrão ouro, já temos a base
        
    dic_ano = parse_dicionario_total(path)
    if not dic_ano:
        continue
        
    target_feats = list(dic_ano.keys())
    target_descs = [str(v) for v in dic_ano.values()]
    
    # Embedding de todas as descrições do ano atual
    target_embs = model.encode(target_descs, convert_to_tensor=True)
    
    # Calcular similaridade entre as Base Features (2023) e TODAS as Target Features (Ano Atual)
    cosine_scores = util.cos_sim(base_embs, target_embs)
    
    # Encontrar a melhor correspondência para cada feature base
    linha_drift = {'Ano': ano}
    
    for i, base_feat in enumerate(base_feats):
        scores = cosine_scores[i]
        best_idx = torch.argmax(scores).item()
        best_score = scores[best_idx].item()
        best_target_feat = target_feats[best_idx]
        
        # Só consideramos um "Match" se a similaridade for razoavelmente alta (> 0.60)
        if best_score > 0.60:
            if best_target_feat == base_feat:
                linha_drift[base_feat] = f"Mantida"
            else:
                linha_drift[base_feat] = f"{best_target_feat} ({best_score:.2f})"
        else:
            linha_drift[base_feat] = f"AUSENTE"
            
    historico_drift.append(linha_drift)

df_drift = pd.DataFrame(historico_drift)
print("Processamento concluído!")


Processamento concluído!


In [5]:
# 4. Tabela Final do Drift Semântico (2009 - 2025)
df_drift.set_index('Ano', inplace=True)
display(df_drift)


,Q006,Q024,TP_ANO_CONCLUIU,TP_LINGUA,TP_SEXO,Q002
Ano,,,,,,
2009,Q21 (0.63),AUSENTE,Q254 (0.85),Q79 (0.92),Mantida,Q18 (0.79)
2010,Q04 (0.90),Q07 (0.64),TP_ST_CONCLUSAO (0.84),Mantida,Mantida,Q03 (0.66)
2011,Q004 (0.90),Q071 (0.78),Mantida,Mantida,Mantida,Q003 (0.86)
2012,Q3 (1.00),Q17 (0.78),Mantida,Mantida,Mantida,Q2 (0.86)
2013,Q003 (0.99),Q017 (0.78),ANO_CONCLUIU (1.00),Mantida,Mantida,Mantida
2014,Q003 (0.99),Q017 (0.78),Mantida,Mantida,Mantida,Mantida
2015,Mantida,Mantida,Mantida,Mantida,Mantida,Mantida
2016,Mantida,Mantida,Mantida,Mantida,Mantida,Mantida
2017,Mantida,Mantida,Mantida,Mantida,Mantida,Mantida
